# Thread Synchronisation

---

## Mutexes

Shorthand for mutual exclusion, a mutex lock is a gate-like object that each thread must take ownership and *lock* before entering the critical section. Once locked, no other thread can take ownership of the mutex, until unlocked by the same thread.

Mutexes are useful for serialising critical sections (i.e. executing them sequentially rather than in parallel).

### Init

To statically initialise a `mutex`:
```c
pthread_mutex_t mutex = PTHREAD_MUTEX_INITIALIZER;
```
For a dynamically allocated `mutex`, use `pthread_mutex_init`:
```c
int pthread_mutex_init(pthread_mutex_t* mutex, const pthread_mutexattr_t* mutexattr);
```
To use this you would call `pthread_mutex_init(&mutex, NULL)`.

### Destroy

To destroy a `mutex`:
```c
int pthread_mutex_destroy(pthread_mutex_t* mutex);
```
Note: dynamically allocated locks still need to be freed.

### Lock/Unlock

To lock:
```c
int pthread_mutex_lock(pthread_mutex_t* mutex);
```
To unlock:
```c
int pthread_mutex_unlock(pthread_mutex_t* mutex);
```
Trylock will attempt to lock, but if it cannot it immediately returns rather than blocking.
```c
int pthread_mutex_trylock(pthread_mutex_t* mutex);
```

---

## Deadlocks

A deadlock is a situation in which two or more threads are stuck waiting trying to access a shared resource held by another, creating a cyclic dependency.

A **self-deadlock** can occur when a thread attempts to acquire a mutex which it has already acquired and has not yet relinquished.

An **ABBA-deadlock** can occur when two threads both try to acquire the lock that the other possesses.

Deadlocks can only occur if all of the four necessary conditions occur:
- **Mutual exclusion:** the resource can be accessed by exactly one thread at a given time
- **Hold and wait:** if a thread both holds a resource and waits to require another
- **No pre-emption:** there is no mechanism to force a thread to relinquish ownership - only the thread holding onto it can release it
- **Circular wait:** a cycle exists in which each thread waits for a resource that is assigned to another thread

---

## Livelocks and Starvation

A livelock happens when multiple threads are kept busy synchronising with each other. There is some exchange in resource, but no progress is made because the exchange forms a cycle.

Starvation is when a thread is never allowed into the critical section because it is perpetually denied access to the lock.

---

## Lock Contention and Granularity

A lock is said to be highly contended if many threads try to acquire it at the same time. The higher the contention, the more parallelism and scalability is limited.

The granularity of a lock indicates the amount of data the lock serialises. There are three levels of granularity:
- Course-grained: protects a large amount of data (large overhead)
- Medium-grained: prodects a moderate amount of data
- Fine-grained: protects an optimally small amount of data (small overhead)

---

## Semaphores

A semaphore is a construct that allows you to control the number of threads that have access to the shared resource. It involves two operations:
- $P(s)$, or `sem_wait`
- $V(s)$, or `sem_post`

A non-negative integer can be set to the semaphore, which limits how many threads can acquire the same resource at the same time. A semaphore with a counter of 1 is known as a binary semaphore, and is essentially the same thing as a mutex.

Semaphore functions are found in `<semaphore.h>`.
```c
int sem_init(sem_t* sem, int pshared, unsigned int value);
```

### Wait

A counter is decremented when the semaphore is acquired. If a thread wants to obtain a semaphore but the counter is at 0, it has to wait until the counter has been incremented.
```c
int sem_wait(sem_t* sem);
```

### Post

Will release a semaphore by incrementing its counter by 1.
```c
int sem_post(sem_t* sem);
```
Which thread is awoken after a semaphore becomes available is random, as it is handled by the kernel.

---

## Semaphores vs. Mutexes

Semaphores are generally more flexible than mutexes. With counting semaphores, we are not just limited to a locked and unlocked state.

While binary semaphores act in much the same way as a mutex, they are distinct in that mutexes have an idea of ownership, whereas semaphores do not. A thread that owns a mutex has to release it, but a thread does not need ownership of the semaphore to post it.

This lets us use semaphores for things like thread synchronisation. We can initialise it to 0 to guarantee that two threads will run at the same time.

Deadlocks can occur with both semaphores and mutexes.